## Make decoding movies

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import os
import numpy as np
import pickle
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# ignore datajoint+jupyter async warnings
import warnings
warnings.simplefilter('ignore', category=DeprecationWarning)
warnings.simplefilter('ignore', category=ResourceWarning)

In [4]:
from spyglass.shijiegu.Analysis_SGU import TrialChoice, DecodeIngredients, DecodeResults2D, ChangeofMindTheta, Imu
from spyglass.shijiegu.decodeHelpers import runSessionNames
from spyglass.utils.nwb_helper_fn import get_nwb_copy_filename
from spyglass.decoding.v0.visualization import make_single_environment_movie
from spyglass.shijiegu.gyroscope import load_tracking_result
from spyglass.shijiegu.helpers import interpolate_to_new_time

In [5]:
from spyglass.shijiegu.Analysis_SGU import RippleTimesWithDecode

### nwb file

In [72]:
nwb_file_name = 'julio20230808.nwb' #('klein20231107.nwb', "10_Rev2Session5", 79),
nwb_copy_file_name = get_nwb_copy_filename(nwb_file_name)

In [73]:
session_interval, position_interval = runSessionNames(nwb_copy_file_name)

### session and time

In [74]:
TrialChoice & {'nwb_file_name':nwb_copy_file_name}

nwb_file_name name of the NWB file,epoch the session epoch for this task and apparatus(1 based),"epoch_name session name, get from IntervalList","choice_reward pandas dataframe, choice"
julio20230808_.nwb,2,02_Seq2Session1,=BLOB=
julio20230808_.nwb,4,04_Seq2Session2,=BLOB=
julio20230808_.nwb,6,06_Seq2Session3,=BLOB=
julio20230808_.nwb,8,08_Seq2Session4,=BLOB=


In [92]:
epoch_num = 2

In [93]:
key={'nwb_file_name':nwb_copy_file_name,'epoch':epoch_num,
     "local_parameter":"dur_0.03_sd_6_hpdFalse"}
print(ChangeofMindTheta & key)
log = ChangeofMindTheta().fetch1_dataframe(key)

session_name = (TrialChoice & key).fetch1('epoch_name')

print(f"Session {session_name}")

*nwb_file_name *epoch    *proportion    *parameter     *local_paramet pandas    
+------------+ +-------+ +------------+ +------------+ +------------+ +--------+
julio20230808_ 2         0.1            params_both_ma dur_0.03_sd_6_ =BLOB=    
 (Total: 1)

Session 02_Seq2Session1


In [94]:
entry = DecodeIngredients & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}
    
# Get data
marks = xr.open_dataset(entry.fetch1('marks'))
position_1d = pd.read_csv(entry.fetch1('position_1d')) #still need 1D position
position_2d = pd.read_csv(entry.fetch1('position_2d')) # need 2D position

entry = DecodeResults2D & {'nwb_file_name':nwb_copy_file_name,
                             'interval_list_name':session_name}

environment_path = entry.fetch1('classifier')
with open(environment_path, 'rb') as file:
    environment2D = pickle.load(file)

decode_path2d = entry.fetch1('posterior')
results = xr.open_zarr(decode_path2d, consolidated=False)

#classifier_path = entry.fetch1('classifier')
#with open(classifier_path, 'rb') as file:
#    classifier = pickle.load(file)

In [95]:
marks_np = np.array(marks.to_dataarray()).squeeze()
timestamps = np.array(position_1d.time)

### Make video for a trial

In [54]:
#log[log.change_of_mind]

In [103]:
trialInd = 67
t0 = log.loc[trialInd,'initial_time'] - 5 #for example 1, use -5.5
t1 = log.loc[trialInd,'timestamp_O']
if np.isnan(t0):
    t0 = t1 - 2

frameToPlot = np.argwhere(np.logical_and(timestamps>=t0,timestamps<=t1)).ravel()
frame0 = frameToPlot[0]
frameLast = frameToPlot[-1]

print("t0",t0)
print("t1",t1)
print("frame0 - frameLast",frameLast - frame0)

t0 1691517690.117429
t1 1691517700.3858564
frame0 - frameLast 5132


### load position inferred from gyroscope data

In [104]:
# output_folder = "/stelmo/shijie/gyro/"
# try:
#     position_ssm = load_tracking_result(output_folder, nwb_file_name, session_name, trialInd)
#     position_ssm_upsample = interpolate_to_new_time(position_ssm, np.array(position_2d.time),
#                                                    upsampling_interpolation_method='nearest')
#     print("Gyroscope-augmented tracking found.")
# except:
#     position_ssm_upsample = position_2d

In [105]:
use_gyro = True
if not use_gyro:
    position_name = (EpochPos & key).fetch1("position_interval")
    position_info = (IntervalPositionInfo() & {
            'nwb_file_name':nwb_copy_file_name,
            'interval_list_name':position_name,
            'position_info_param_name':'default'}).fetch1_dataframe()
else:
    imu_name = "big_acc_bias"
    key_imu={'nwb_file_name':nwb_copy_file_name,
                  'epoch':epoch_num,
                  'trial':trialInd,
                  "parameter":imu_name}
    position_info = Imu().fetch1_dataframe(key_imu)

position_ssm_upsample = interpolate_to_new_time(position_info, np.array(position_2d.time),
                                                upsampling_interpolation_method='nearest')

### Make video

In [99]:
import numpy as np
import matplotlib.pyplot as plt
from spyglass.shijiegu.parallel_video_writer import create_parallel_video, VideoConfig
from spyglass.shijiegu.parallel_decode_video import make_single_environment_movie

In [100]:
np.min(position_ssm_upsample.head_position_y)

46.46434020996094

In [101]:
np.max(position_ssm_upsample.head_position_y)

234.1004180908203

In [106]:
make_single_environment_movie(
    slice(frame0, frameLast),
    environment2D,
    results,
    position_ssm_upsample,
    marks_np,
    movie_name=f"{nwb_copy_file_name[:13]}_session{session_name}_trial{trialInd}_decode_ssm.mp4",
    sampling_frequency=500,
    video_slowdown=12,
    position_name=["head_position_x", "head_position_y"],
    direction_name="head_orientation",
    vmax=0.07,
    max_workers = 20,
)

finished setting up
[parallel] Using 20 workers for 5132 frames
[parallel] Writing frames to /tmp/parallel_video_frames_lkxwi3ax
[parallel] frame 0
finished setting up
starting to render frame 0
starting to render frame 1
starting to render frame 2
starting to render frame 3
finished setting up
starting to render frame 257
starting to render frame 4
starting to render frame 5
starting to render frame 258
starting to render frame 6
starting to render frame 259
starting to render frame 7
starting to render frame 260
starting to render frame 8
finished setting up
starting to render frame 514
starting to render frame 261
starting to render frame 9
starting to render frame 262
starting to render frame 515
starting to render frame 10
starting to render frame 263
starting to render frame 516
starting to render frame 11
starting to render frame 264
starting to render frame 517
starting to render frame 12
finished setting up
starting to render frame 771
starting to render frame 265
starting to 

1